# Qwen2.5-3B-Instruct — DAPO
Starts from **Qwen/Qwen2.5-3B-Instruct**, pinned to a model commit.
Full-parameter training on one **RTX PRO 6000 Blackwell 96GB**. This notebook
runs DAPO training only. It does not load EXAONE, an SFT checkpoint or a LoRA adapter.

Settings in `configs/dapo_qwen25_3b.yaml`:
- G=8, 16 retained mixed-correctness questions (128 responses) per rollout.
- **Two disjoint 64-response minibatches per rollout**, with fixed old-policy probabilities
  and advantages. Each response is used once (`num_iterations=1`).
- 300 optimizer updates = 150 fresh rollout batches; checkpoints every 20 updates.
- LR 1e-6, 20-update warmup then constant LR; temperature/top-p 1; clip 0.20/0.28.
- Rule-based +1/-1 reward, group normalization, token-level DAPO loss, no KL.
- Maximum response 20,480 tokens; soft length penalty begins at 16,384. No budget forcing.

Uses the same verified 14,068-row decontaminated English DAPO dataset and English
instruction. Qwen's native chat template is used. `<|im_end|>` stops generation;
`<|endoftext|>` is padding and is excluded from the configured stop list.
FP32 parameters/Adam state, BF16 compute/rollouts, gradient checkpointing and microbatch 1.
vLLM generates up to 16 concurrent responses and sleeps during optimization.

Preparation, GPU smoke and full training are separate cells. Local CPU tests do not
establish GPU memory fit: run smoke on the target GPU first. Fifteen full resumable
checkpoints plus smoke need roughly **650 GB** of Drive space. All settings below
are plain Python variables. This is a new experiment; EXAONE continuation is separate.

Code, config and dependency locks are bundled. No GitHub push or clone is needed.
The memory fix checkpoints the vocabulary projection together with log-probabilities
in 128-token chunks, avoiding full 20K-token vocabulary logits during backward.
Open this updated notebook in a fresh GPU runtime. GPU smoke still verifies runtime fit.
Defaults recover the interrupted run from checkpoint 40 into a separate Drive folder.

In [ ]:
OUTPUT_ROOT = "/content/drive/MyDrive/LG-AIME-DAPO-Qwen25-3B-MiniBatch300-MemoryFix"
RECOVER_FROM_CHECKPOINT = "/content/drive/MyDrive/LG-AIME-DAPO-Qwen25-3B-MiniBatch300/checkpoints/dapo_qwen25_3b_base/checkpoint-40"
# For a new run from the original model, set RECOVER_FROM_CHECKPOINT = "".
CODE_PARENT = "/content/lg-aime-dapo-qwen25"
RL_ENV = "/content/lg-dapo-env"
WORK_DIR = "/content/qwen25-dapo-work"
CONFIG = "configs/dapo_qwen25_3b.yaml"

## Setup
Enable HF_TOKEN in Colab Secrets for the dataset. The published Qwen model is
loaded directly; no prior SFT or EXAONE checkpoint is needed. Code is installed
in a versioned local folder and recorded as a reproducible Git snapshot so that
the shared trainer can verify exact checkpoint-resume identity.

In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
BUNDLE = '...'
bundle_bytes = base64.b64decode(BUNDLE)
bundle_id = hashlib.sha256(bundle_bytes).hexdigest()
CODE_ROOT = str(Path(CODE_PARENT) / bundle_id[:16])
Path(CODE_ROOT).mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as bundle:
    for name in bundle.namelist():
        path = Path(CODE_ROOT) / name
        data = bundle.read(name)
        if path.exists() and path.read_bytes() != data:
            raise ValueError(f"Bundled file changed: {path}. Use a fresh CODE_PARENT.")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
def git(*args, **kwargs):
    return subprocess.check_output(["git", "-C", CODE_ROOT, *args], text=True, **kwargs).strip()
if not (Path(CODE_ROOT) / ".git").exists():
    git("init", "--quiet")
    git("config", "core.autocrlf", "false")
    git("config", "core.filemode", "false")
    git("add", ".")
    commit_env = dict(os.environ, GIT_AUTHOR_NAME="Notebook bundle", GIT_COMMITTER_NAME="Notebook bundle",
                      GIT_AUTHOR_EMAIL="bundle@localhost", GIT_COMMITTER_EMAIL="bundle@localhost",
                      GIT_AUTHOR_DATE="2026-01-01T00:00:00+00:00", GIT_COMMITTER_DATE="2026-01-01T00:00:00+00:00")
    git("-c", "commit.gpgsign=false", "commit", "--quiet", "-m", "Bundled DAPO " + bundle_id,
        env=commit_env)
if git("status", "--porcelain"):
    raise ValueError("Bundled code was modified; use a fresh CODE_PARENT.")
print("Bundled code:", CODE_ROOT, "commit:", git("rev-parse", "HEAD"))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.22"])
subprocess.check_call([sys.executable, "scripts/setup_dapo_runtime.py", "--venv", RL_ENV,
                       "--architecture", "Qwen2ForCausalLM"], cwd=CODE_ROOT)
sys.path.insert(0, CODE_ROOT)
from common.process import run_logged

## Paths and training command
Full checkpoints and logs are isolated from all previous EXAONE and Qwen SFT runs.

In [ ]:
from train.dapo_data import load_config, batch_schedule
cfg = load_config(Path(CODE_ROOT) / CONFIG)
if cfg["model"]["repo"] != "Qwen/Qwen2.5-3B-Instruct":
    raise ValueError("Use the Qwen2.5-3B-Instruct config")
RUN_NAME = cfg["run_name_prefix"] + "_base"
run_logs = Path(OUTPUT_ROOT) / "logs" / RUN_NAME
checkpoint_root = Path(OUTPUT_ROOT) / "checkpoints" / RUN_NAME
TRAIN_COMMAND = [str(Path(RL_ENV) / "bin/python"), "train/run_dapo.py", "--config", CONFIG,
                 "--model-kind", "base", "--output-root", OUTPUT_ROOT, "--work-dir", WORK_DIR]
print("Starting model:", cfg["model"]["repo"])
print("Updates per rollout:", batch_schedule(cfg)["updates_per_rollout"])
print("Checkpoints:", checkpoint_root)
print("Training logs:", run_logs)

## Prepare and inspect the prompt
Checks dataset provenance, tokenizer access, native stop/padding tokens and prompt lengths.
Does not load model weights. Saves resolved configuration and the data report to Drive.

In [ ]:
run_logged(TRAIN_COMMAND + ["--prepare-only"], cwd=CODE_ROOT,
           log_path=run_logs / "prepare_console.log")

## GPU smoke — two updates on one rollout
Uses G=8, two retained mixed groups (16 responses), two disjoint 8-response minibatches,
the full token cap, and no warmup.
Verifies generation, rewards, minibatch updates and checkpoint saving on your GPU.
Smoke has its own timestamped directory; its weights are never used for the full run.
The rows must have minibatch_index 1 then 2, with policy_iteration=1 in both.
The second row has cached_rollout=true, reused_rollout=false and new_generated_tokens=0.
Clipping can still be zero if the update is small. A nonzero clip fraction is not required.

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    from datetime import datetime, timezone
    smoke_root = Path(OUTPUT_ROOT) / "smoke_attempts" / datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
    command = list(TRAIN_COMMAND)
    command[command.index("--output-root") + 1] = str(smoke_root)
    run_logged(command + ["--smoke"], cwd=CODE_ROOT,
               log_path=smoke_root / "smoke_console.log", compact_progress=True)
    smoke_log = smoke_root / "logs" / (RUN_NAME + "_smoke") / "steps.jsonl"
    records = [json.loads(line) for line in smoke_log.read_text().splitlines()]
    assert [r["minibatch_index"] for r in records] == [1, 2]
    assert [r["policy_iteration"] for r in records] == [1, 1]
    assert not any(r["reused_rollout"] for r in records)
    assert records[-1]["cached_rollout"] and records[-1]["new_generated_tokens"] == 0
    print("Two-update smoke completed. Logs:", smoke_log)
else:
    print("Set RUN_SMOKE = True to run the separate GPU smoke.")

## Recover training — 300 total optimizer updates

Defaults load the old **checkpoint 40**, including optimizer, scheduler and RNG,
and continue with update 41 under the new `OUTPUT_ROOT`. Updates 41–43 from the
interrupted run are replayed. The original checkpoints/logs are preserved.
The total remains 300 updates, with the same data, learning rate, rewards,
64-response minibatches, and 20,480-token cap. The implementation changes, so the
recovered run records its new code identity and the parent checkpoint explicitly.

Leave `RECOVER_FROM_CHECKPOINT` set when resuming this recovery run. A completed
checkpoint in the new folder is selected automatically on rerun. `RESUME_CHECKPOINT`
can explicitly select the latest complete checkpoint instead. If the runtime stops
before saving checkpoint 60, rerun from parent checkpoint 40 in this same recovery folder.
Use a fresh GPU runtime after the CUDA error, then run setup and preparation again.

In [ ]:
RUN_TRAINING = False
RESUME_CHECKPOINT = ""
if RUN_TRAINING:
    command = list(TRAIN_COMMAND)
    if RECOVER_FROM_CHECKPOINT:
        command += ["--recover-from-checkpoint", RECOVER_FROM_CHECKPOINT]
    complete = sorted(
        (p for p in checkpoint_root.glob("checkpoint-*") if (p / "dapo_checkpoint.json").is_file()),
        key=lambda p: int(p.name.split("-")[-1]))
    latest = RESUME_CHECKPOINT or (str(complete[-1]) if complete else "")
    if latest:
        command += ["--resume-from-checkpoint", latest]
    if complete and int(complete[-1].name.split("-")[-1]) >= cfg["training"]["max_steps"]:
        print("The full 300-update run is already complete:", complete[-1])
    else:
        print("Continue from:", latest or RECOVER_FROM_CHECKPOINT or "original model")
        run_logged(command, cwd=CODE_ROOT, log_path=run_logs / "training_console.log",
                   compact_progress=True)
else:
    print("Set RUN_TRAINING = True to recover/continue training.")

## Drive outputs
Under OUTPUT_ROOT:
- `checkpoints/dapo_qwen25_3b_base/checkpoint-<step>/`: full resumable checkpoints.
- `logs/dapo_qwen25_3b_base/steps.jsonl`: loss, gradient norm, learning rate, reward,
  entropy/clipping, candidate accuracy, acceptance counts, length and timing metrics.
- `logs/dapo_qwen25_3b_base/rollouts/attempt_*/update_*/`: raw answers and scoring records.
- `logs/dapo_qwen25_3b_base/`: console logs, resolved config, data report and run manifest.
- `smoke_attempts/<timestamp>/`: separate smoke logs and checkpoint.

`rollout_first_update`, `minibatch_index` and `policy_iteration` identify each update. Rollout statistics
repeat on the second pass; sum `new_generated_tokens` for actual new generation.
Cached rollout accuracy is not a fresh measurement of the updated model.
Console logs are automatic. Save the notebook itself in Drive through Colab if you
also want its displayed cell outputs. No checkpoint weights are packaged for sharing.